# Strategy - Correlation Regime

In [ ]:
# ============================================================
# S08 - Correlation Regime Strategy
# ============================================================

import xarray as xr
import numpy as np

import qnt.data as qndata
import qnt.ta as qnta
import qnt.output as qnout

# ============================================================
# Load Data
# ============================================================

data = qndata.cryptodaily_load_data(
    min_date="2015-01-01"
)

# ============================================================
# Strategy
# ============================================================

def strategy(data):

    # --------------------------------------------------------
    # Extract Data
    # --------------------------------------------------------

    close = data.sel(field="close")
    is_liquid = data.sel(field="is_liquid")

    close = xr.where(
        is_liquid == 1,
        close,
        np.nan
    )

    # --------------------------------------------------------
    # Daily Returns
    # --------------------------------------------------------

    ret = close / close.shift(time=1) - 1

    # --------------------------------------------------------
    # Correlation Regime Proxy
    # --------------------------------------------------------

    cross_sectional_dispersion = ret.std("asset")

    dispersion_ma60 = qnta.sma(
        cross_sectional_dispersion,
        60
    )

    low_correlation_regime = xr.where(
        cross_sectional_dispersion > dispersion_ma60,
        1,
        0
    )

    # --------------------------------------------------------
    # Bitcoin Data
    # --------------------------------------------------------

    btc = data.sel(
        field="close",
        asset="BTC"
    )

    # --------------------------------------------------------
    # Momentum Measures
    # --------------------------------------------------------

    asset_mom = (
        close /
        close.shift(time=21)
        - 1
    )

    btc_mom = (
        btc /
        btc.shift(time=21)
        - 1
    )

    # --------------------------------------------------------
    # Relative Strength vs BTC
    # --------------------------------------------------------

    relative_strength = (
        asset_mom
        - btc_mom
    )

    score = relative_strength.rank(
        "asset"
    )

    # --------------------------------------------------------
    # Volatility Estimate
    # --------------------------------------------------------

    vol20 = qnta.std(
        ret,
        20
    )

    # --------------------------------------------------------
    # Select Leaders
    # --------------------------------------------------------

    ranks = score.rank(
        "asset"
    )

    top_assets = xr.where(
        ranks >= 6,
        1,
        0
    )

    weights = score * top_assets

    # --------------------------------------------------------
    # Inverse Volatility Weighting
    # --------------------------------------------------------

    inv_vol = (
        1 /
        (vol20 + 1e-6)
    )

    weights = weights * inv_vol

    # --------------------------------------------------------
    # Normalize
    # --------------------------------------------------------

    denom = weights.sum(
        "asset"
    )

    weights = xr.where(
        denom > 0,
        weights / denom,
        0
    )

    # --------------------------------------------------------
    # Apply Correlation Regime Filter
    # --------------------------------------------------------

    weights = weights * low_correlation_regime

    return weights.fillna(0)

# ============================================================
# Generate Portfolio Weights
# ============================================================

weights = strategy(data)

# ============================================================
# Clean Output
# ============================================================

weights = qnout.clean(
    weights,
    data,
    "crypto_daily_long"
)

# ============================================================
# Write Submission Output
# ============================================================

qnout.write(weights)

100% (15259192 of 15259192) |############| Elapsed Time: 0:00:00 Time:  0:00:000:00
Output cleaning...
Fix unique timestamps
Forward filling missing prices...
Check liquidity...
Ok.
Check for missed dates...
Ok.
Check positive positions...
Ok.
Final normalization...
Output cleaning complete.
Write output: /root/fractions.nc.gz

In [ ]:
# ============================================================
# Strategy Statistics
# ============================================================

import qnt.stats as qnstats

stats = qnstats.calc_stat(
    data,
    weights.sel(time=slice("2016-01-01", None))
)

stats_pd = stats.to_pandas()

print(stats_pd.tail())

print("\nFinal Metrics:")

print(
    stats_pd.iloc[-1][[
        "equity",
        "sharpe_ratio",
        "max_drawdown",
        "avg_turnover",
        "avg_holding_time"
    ]]
)

field           equity  relative_return  volatility  underwater  max_drawdown  \
time                                                                            
2026-06-05  194.196715        -0.040417    0.579773   -0.386664     -0.811739   
2026-06-06  192.444443        -0.009023    0.579707   -0.392199     -0.811739   
2026-06-07  196.486756         0.021005    0.579661   -0.379432     -0.811739   
2026-06-08  195.694818        -0.004030    0.579588   -0.381933     -0.811739   
2026-06-09  192.869041        -0.014440    0.579534   -0.390858     -0.811739   

field       sharpe_ratio  mean_return  bias  instruments  avg_turnover  \
time                                                                     
2026-06-05      1.146751     0.664855   1.0         32.0      0.419105   
2026-06-06      1.145090     0.663817   1.0         32.0      0.419006   
2026-06-07      1.148350     0.665654   1.0         32.0      0.418908   
2026-06-08      1.147528     0.665094   1.0         32.0      0.418812   
2026-06-09      1.144949     0.663537   1.0         32.0      0.418727   

field       avg_holding_time  
time                          
2026-06-05          1.780225  
2026-06-06          1.780339  
2026-06-07          1.780456  
2026-06-08          1.780682  
2026-06-09          1.789708  

Final Metrics:
field
equity              192.869041
sharpe_ratio          1.144949
max_drawdown         -0.811739
avg_turnover          0.418727
avg_holding_time      1.789708
Name: 2026-06-09 00:00:00, dtype: float64